In [1]:
import os
import sys
# Agregar el directorio raíz del proyecto al path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
    
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *   
from services.utilidades import *
from services.comprobar_netcdf import *

In [2]:
# Recargar módulos locales (ejecutar esta celda cuando cambies código en services/)
import importlib
import sys

# Lista de módulos locales a recargar
modulos_locales = [
    'services.carga_de_datos',
    'services.DataFrame_maker',
    'services.prueba',
    'services.utilidades',
    'services.get_tspan',
    'services.polar2uv',
    'services.uv2polar'
]

for modulo in modulos_locales:
    if modulo in sys.modules:
        importlib.reload(sys.modules[modulo])
        print(f"✓ Recargado: {modulo}")
    else:
        print(f"- No cargado: {modulo}")

# Re-importar después de recargar
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *

print("\n¡Módulos locales recargados!")

✓ Recargado: services.carga_de_datos
✓ Recargado: services.DataFrame_maker
- No cargado: services.prueba
✓ Recargado: services.utilidades
✓ Recargado: services.get_tspan
✓ Recargado: services.polar2uv
✓ Recargado: services.uv2polar

¡Módulos locales recargados!


In [3]:
# %matplotlib tk

In [4]:
dict_de_boyas = {}

### 2. OLEAJE

In [5]:
# # Archivo .dat de oleaje
# nombre = "BOT1-11-T100_DAT"
# nombre_de_archivo = "BOT1-11-T100-MENSAJE3-TRIAXYSG3-NS09276-Z0-REALT-20251119-20260310-RAW.dat"
# ruta_a_carpeta = "C:\\Users\\Atmosfera\\Desktop\\datos_procesados\\revisiones\\2026\\marzo\\BOT1-11-T100\\DATOS"
# indices = Crear_indices_para_crudos_dat_o_txt("realt").get_indices()
# df_dat = cargar_telemetria_oleaje_dat(ruta_a_carpeta, nombre_de_archivo, indices)
# dict_de_boyas[nombre] = df_dat # Guardo el dataframe dentro del diccionario de boyas.
# df_dat.head(2)

In [6]:
# p1: Crudo unido men
nombre = "BOT1-02-T80_pkl_realt"
archivos_csv = "msj3_crudo_unido_mem_2025111619_2026020821.pkl"
ruta_a_carpeta = "C:\\Users\\Atmosfera\\Desktop\\datos_procesados\\revisiones\\2026\\marzo\\BOT1-02-T80\\DATOS"
df_pkl_crudo_realt = cargar_pickle_oleaje(ruta_a_carpeta, archivos_csv)
dict_de_boyas[nombre] = df_pkl_crudo_realt # Guardo el dataframe dentro del diccionario de boyas.
df_pkl_crudo_realt.head(2)

,tspan,Hs,Hmax,Tp,dir,vardir
0,2025-11-16 19:00:00,0.58,1.01,3.3,73.0,36.0
1,2025-11-16 20:00:00,0.52,0.84,3.4,69.0,38.0


In [7]:
# 1.5. p1: Cargar NETCDF de corrientes 
nombre = "BOT1-02-T80_NC"
nombre_de_archivo = "BOT1-02-T80-TRIAXYSG3-NS09171-Z0-MEM-20251116-20260208.nc"
# ruta_a_carpeta = "C:\\Users\\Atmosfera\\Desktop\\datos_procesados\\revisiones\\2026\\marzo\\BOT1-02-T80\\DATOS"
ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Final\\Reporte_1.3\\2026\\Marzo\\PEMEX-ASM-CICESE-658225821-149\\RECUPERACION\\BOT1-02-T80\\OLEAJE\\VALIDADOS"
df_nc, nc, DirSpec = cargar_nc_oleaje(ruta_a_carpeta, nombre_de_archivo)
dict_de_boyas[nombre] = df_nc # Guardo el dataframe dentro del diccionario de boyas.
df_nc.head(2)


,tspan,Hs,Hmax,Tp,dir,vardir
0,2025-11-16 19:00:00,0.58,1.01,3.3,73.0,36.0
1,2025-11-16 20:00:00,0.52,0.84,3.4,69.0,38.0


In [8]:
# Otras revisiones del NC:
# comprobar_nombre_nc
comprobar_nombre_nc(nombre_de_archivo, nc)
comprobar_porcentaje_de_variable(nc, ruta_a_carpeta, "BOT1-03-T50_datos_word_2025111717_2025123001.xlsx", variables="oleaje")

NAM correcto


,Variable,esperados,validados,porcentaje_validados
0,Hs,2019,2019.0,100.0
1,Hm,2019,2019.0,100.0
2,Tp,2019,2019.0,100.0
3,dir,2019,2019.0,100.0
4,vardir,2019,2019.0,100.0
5,dirspec,2019,2019.0,100.0


In [9]:
# Crear el diccionario de boyas con los dataframes cargados
df = crear_dataframe_de_boyas(dict_de_boyas)

Index(['tspan', 'Hs_BOT1-02-T80_pkl_realt', 'Hmax_BOT1-02-T80_pkl_realt',
       'Tp_BOT1-02-T80_pkl_realt', 'dir_BOT1-02-T80_pkl_realt',
       'vardir_BOT1-02-T80_pkl_realt', 'Hs_BOT1-02-T80_NC',
       'Hmax_BOT1-02-T80_NC', 'Tp_BOT1-02-T80_NC', 'dir_BOT1-02-T80_NC',
       'vardir_BOT1-02-T80_NC'],
      dtype='str')


In [10]:
app = InteractivePlotterTk(df)
# Iniciar el bucle de la aplicación
app.root.mainloop()

# NOTAS PENDIENTES:
# - Arreglar que al eliminar puntos para que no tenga límite el historial de deshacer
# - Arreglar para que solo se grafiquen las columnas elegidas por el usuario
# - Agregar una función que grafique las variables de Rap y Dir --> sería algo como que df.iloc[idx,col][prof]
# - Arreglar el caso de cuando el usuario elimina puntos de Rap o Dir colocar U y V como nans --> df.iloc[idx,col][prof] = np.nan
# - Agregar la funcionalidad de guardar el DataFrame corregido en un archivo .csv o .xlsx, pkl
# - Agregar la funcionalidad de devolver puntos eliminados (no deshacer la última acción, sino devolver puntos específicos)

In [11]:
# idxs = app.deletion_log["vardir_BOT1-04-T80_pkl_realt_V"]["idxs"]
# tspan_num = mdates.num2date(app.data.loc[idxs,"tspan"])
# tspan_pd = pd.to_datetime(tspan_num)
# df_out = pd.DataFrame({"tspan": tspan_pd})
# df_out.to_csv("tspan_eliminados.csv", index=False)

